# Model initiation

In [1]:
from extraction_tools import *
from models import *
from glob import glob

save_dir = r'D:\AIC_NEW_EXTRACTION'
keyframe_extraction_model = TransNetV2Model(device="cuda")
keyframe_feature_extraction_model = SigLIP2Model(name="google/siglip2-so400m-patch14-384", device="cuda")
transcription_extraction_model = FasterWhisperModel(name="turbo", device="cuda")
transcription_feature_extraction_model = BGEM3Model(name="BAAI/bge-m3", device="cuda")

# keyframe_extractor = KeyframeAndFeatureExtractor(
#     keyframe_extraction_model,
#     keyframe_feature_extraction_model,
#     save_dir
# )
transcription_extractor = TranscriptFeatureProcessor(
    transcription_extraction_model,
    transcription_feature_extraction_model,
    save_dir
)


c:\Users\Bui Thien Nghia\Documents\PERSONAL FILES 2\GITHUB_REPOS\Determinative-HCMCAIC26\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[INFO] Loading quantized turbo
[INFO] turbo is on cuda device
[INFO] Loading quantized BAAI/bge-m3
[INFO] BAAI/bge-m3 is on cuda device


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2152.59it/s]


# Extract keyframes

In [ ]:
vid_paths = glob(fr"")

keyframe_extractor.extract_and_save_keyframes(
    vid_paths,
    feature_extraction_batch_size=32,
    shot_confidence_threshold=0.7,
    clustering_similarity_threshold=0.9,
    max_workers=4
)

# Extract transcripts

In [2]:
vid_paths = glob(fr"E:/AIC_VID_ONLY/vids/*/*.mp4")
map_paths = glob(fr"D:/AIC_NEW_EXTRACTION/maps/*.csv")

transcription_extractor.extract_and_save_transcript(
    vid_paths,
    map_paths,
    transcription_batch_size=16,
    transcription_language="vi",
    alignment_max_time_distance=10,
    feature_extraction_batch_size=32,
    max_workers=4
)

Extracting transcripts from video:   0%|          | 0/1471 [00:00<?, ?it/s]


pre tokenize: 100%|██████████| 6/6 [00:00<00:00, 280.40it/s]















pre tokenize: 100%|██████████| 6/6 [00:00<00:00, 805.54it/s]








Inference Embeddings: 100%|██████████| 5/5 [00:01<00:00,  4.11it/s]

Inference Embeddings: 100%|██████████| 5/5 [00:01<00:00,  4.09it/s]



Inference Embeddings: 100%|██████████| 6/6 [00:01<00:00,  4.28it/s]






pre tokenize: 100%|██████████| 5/5 [00:00<00:00, 687.46it/s]01:19<7:05:40, 17.40s/it] 

pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 740.13it/s]




pre tokenize: 100%|██████████| 6/6 [00:00<00:00, 1011.20it/s]






Inference Embeddings: 100%|██████████| 5/5 [00:00<00:00,  5.18it/s]




Inference Embeddings: 100%|██████████| 4/4 [00:00<00:00,  4.74it/s]








pre tokenize: 100%|██████████| 6/6 [00:00<00:00, 831.68it/s]02:11<10:12:50, 25.08s/it]

pre tokenize: 100%|██████████| 5/5 [00:00<00:00, 849.63it/s]




pre tokenize: 100%|██████████| 6/6 [00:00

# Test

In [1]:
from pathlib import Path
from tqdm import tqdm
from PIL import Image
from extraction_tools import BEiT3Model

img_paths = [str(path) for path in list(Path(r"D:\AIC_NEW_EXTRACTION").glob("imgs/**/*.jpg"))]

beit3_model = BEiT3Model("timm/beit3_large_patch16_224.pt", device="cuda")
beit3_model.load()

[INFO] Loading quantized timm/beit3_large_patch16_224.pt
[INFO] timm/beit3_large_patch16_224.pt is on cuda device


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

[transformers] Requested torchvision backend is not available. Falling back to pil backend.


In [ ]:
batch_size = 1024
feats = []
for i in tqdm(list(range(0, len(img_paths), batch_size))):
    j = i + batch_size if i + batch_size < len(img_paths) else len(img_paths)
    imgs = [Image.open(path) for path in img_paths[i:j]]
    feat = beit3_model.run(imgs, batch_size=batch_size)
    for f in feat: feats.append(f)

In [ ]:
import os
feat_dir = Path(r"D:/AIC_NEW_EXTRACTION/feats/beit3_dense/")
for path, feat in zip(img_paths, feats):
    path = Path(path)
    if not os.path.exists(os.path.join(feat_dir, path.parts[-2])):
        os.mkdir(os.path.join(feat_dir, path.parts[-2]))
    np.save(str(Path(feat_dir) / path.parts[-2] / path.parts[-1]), feat)